# Chat History in Redis — Hands-On

**TechBot scenario:** Build the memory layer that remembers what the user said earlier in the conversation.

**Prerequisites:**
```bash
docker run -d --name redis-stack -p 6379:6379 -p 8001:8001 redis/redis-stack:latest
pip install redis langchain-redis langchain-openai redisvl
```

## Part 1: Raw redis-py Implementation

Build the chat history from scratch using only redis-py. This gives you full control and helps you understand exactly what is stored.

In [ ]:
import redis
import json
import time
import uuid

r = redis.Redis(host="localhost", port=6379, decode_responses=True)
print("Connected:", r.ping())

In [ ]:
class ChatHistory:
    """
    Stores conversation history in Redis using:
      - A List for the ordered message sequence
      - A Hash for session metadata
      - Sliding TTL: resets on every new message
    """
    
    def __init__(self, session_id: str, redis_client, ttl: int = 86400, max_messages: int = 100):
        self.session_id = session_id
        self.r = redis_client
        self.ttl = ttl
        self.max_messages = max_messages
        self.messages_key = f"chat:{session_id}:messages"
        self.meta_key = f"chat:{session_id}:metadata"
    
    def init_session(self, user_id: str, model: str, system_prompt: str):
        """Create session metadata. Safe to call multiple times (HSET is idempotent)."""
        self.r.hset(self.meta_key, mapping={
            "user_id":       user_id,
            "model":         model,
            "system_prompt": system_prompt,
            "created_at":    str(int(time.time())),
            "message_count": "0"
        })
        self.r.expire(self.meta_key, self.ttl)
    
    def add_message(self, role: str, content: str):
        """Append a message and reset the sliding TTL."""
        msg = json.dumps({"role": role, "content": content, "ts": int(time.time())})
        self.r.rpush(self.messages_key, msg)
        self.r.ltrim(self.messages_key, -self.max_messages, -1)  # cap history
        self.r.expire(self.messages_key, self.ttl)               # sliding TTL
        self.r.hincrby(self.meta_key, "message_count", 1)
        self.r.expire(self.meta_key, self.ttl)
    
    def get_messages(self, last_n: int = 20) -> list:
        """Get the last N messages for LLM context."""
        raw = self.r.lrange(self.messages_key, -last_n, -1)
        return [json.loads(m) for m in raw]
    
    def get_metadata(self) -> dict:
        return self.r.hgetall(self.meta_key)
    
    def build_llm_messages(self, new_user_message: str, window: int = 10) -> list:
        """Build the messages list to send to an LLM (includes system prompt + history + new message)."""
        meta = self.get_metadata()
        messages = [{"role": "system", "content": meta.get("system_prompt", "You are a helpful assistant.")}]
        messages += self.get_messages(last_n=window * 2)  # *2 for user+assistant pairs
        messages.append({"role": "user", "content": new_user_message})
        return messages
    
    def clear(self):
        self.r.delete(self.messages_key, self.meta_key)


# ---- Simulate TechBot session ----

session_id = f"user-alice-{uuid.uuid4().hex[:8]}"
history = ChatHistory(session_id, r, ttl=3600, max_messages=50)

history.init_session(
    user_id="alice@techcorp.com",
    model="llama3-8b",
    system_prompt="You are TechBot, a helpful technical support assistant."
)

# Simulate conversation
history.add_message("user",      "How do I install the SDK?")
history.add_message("assistant", "Run: pip install techbot-sdk")
history.add_message("user",      "What about on Windows?")
history.add_message("assistant", "Same command — pip install techbot-sdk works on Windows too.")
history.add_message("user",      "And how do I authenticate?")

print(f"Session: {session_id}")
print(f"Metadata: {history.get_metadata()}")
print(f"\nAll messages ({len(history.get_messages())} total):")
for msg in history.get_messages():
    print(f"  [{msg['role']}]: {msg['content']}")

In [ ]:
# Build the full prompt that would be sent to the LLM
# (The LLM needs the context to understand what "authenticate" refers to)
llm_messages = history.build_llm_messages(
    new_user_message="And how do I authenticate?",
    window=5
)

print("Messages to send to LLM:")
for m in llm_messages:
    print(f"  [{m['role']}]: {m['content'][:80]}..." if len(m['content']) > 80 else f"  [{m['role']}]: {m['content']}")

## Part 2: LangChain's RedisChatMessageHistory

LangChain abstracts the above into a ready-made class that stores messages as JSON documents with a RediSearch index — enabling filtering by session, time range, or message type.

In [ ]:
from langchain_redis import RedisChatMessageHistory

# Create (or reconnect to) a session
lc_session_id = f"lc-alice-{uuid.uuid4().hex[:8]}"

lc_history = RedisChatMessageHistory(
    session_id=lc_session_id,
    redis_url="redis://localhost:6379",
    ttl=3600,
    key_prefix="chat:",
    index_name="idx:chat_history"
)

# Add messages (LangChain uses typed Message objects internally)
lc_history.add_user_message("How do I install the SDK?")
lc_history.add_ai_message("Run: pip install techbot-sdk")
lc_history.add_user_message("What about on Windows?")
lc_history.add_ai_message("Same command — works on Windows too.")

# Retrieve — returns list of LangChain BaseMessage objects
messages = lc_history.messages
print(f"LangChain session '{lc_session_id}' has {len(messages)} messages:")
for msg in messages:
    msg_type = msg.__class__.__name__.replace("Message", "")
    print(f"  [{msg_type}]: {msg.content}")

In [ ]:
# Inspect what was actually stored in Redis
# LangChain stores each message as a separate JSON document
print("Keys created by LangChain:")
for key in r.scan_iter(f"chat:{lc_session_id}*"):
    key_type = r.type(key)
    print(f"  {key}  [{key_type}]")
    if key_type == "ReJSON-RL":  # JSON type
        # Note: viewing JSON keys requires the JSON.GET command
        # In RedisInsight GUI this is automatic
        pass

## Part 3: Wiring LangChain History into a Chat Loop

Use `RunnableWithMessageHistory` to automatically load/save history on every LLM call.

We'll use a **fake LLM** so this works without any API keys.

In [ ]:
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import AIMessage
from langchain_core.runnables import RunnableLambda

# Fake LLM — echoes back the last user message with a prefix
# Replace this with: ChatOpenAI(model="gpt-4o-mini") or your vLLM client
def fake_llm(messages):
    last_user = next(
        (m.content for m in reversed(messages.messages) if hasattr(m, 'type') and m.type == 'human'),
        "(no user message)"
    )
    return AIMessage(content=f"[TechBot] You asked: '{last_user}'. Here is my answer.")

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are TechBot, a helpful technical support assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# Build the chain: prompt | fake_llm
chain = prompt | RunnableLambda(fake_llm)

# Wrap with history management
def get_session_history(session_id: str) -> RedisChatMessageHistory:
    return RedisChatMessageHistory(
        session_id=session_id,
        redis_url="redis://localhost:6379",
        ttl=3600
    )

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

chat_session = "demo-session-001"

# Each invoke automatically loads history, appends new messages, saves to Redis
turns = [
    "How do I install the SDK?",
    "What about on Windows?",
    "And how do I authenticate?"
]

for question in turns:
    response = chain_with_history.invoke(
        {"input": question},
        config={"configurable": {"session_id": chat_session}}
    )
    print(f"User: {question}")
    print(f"Bot:  {response.content}")
    print()

# Verify it was saved to Redis
saved = get_session_history(chat_session)
print(f"\nMessages saved in Redis: {len(saved.messages)}")

## Part 4: Multi-Session Isolation Test

Verify that Alice's and Bob's sessions don't interfere with each other.

In [ ]:
alice_session = ChatHistory("alice-multi-test", r, ttl=300)
bob_session   = ChatHistory("bob-multi-test",   r, ttl=300)

alice_session.init_session("alice", "llama3", "You are TechBot.")
bob_session.init_session("bob", "llama3", "You are TechBot.")

alice_session.add_message("user",      "Alice: How do I reset my password?")
alice_session.add_message("assistant", "Go to Settings > Security > Reset Password")

bob_session.add_message("user",      "Bob: How do I export my data?")
bob_session.add_message("assistant", "Go to Settings > Export > Download CSV")

print("Alice's history:")
for m in alice_session.get_messages():
    print(f"  {m['content']}")

print("\nBob's history:")
for m in bob_session.get_messages():
    print(f"  {m['content']}")

# Verify isolation — alice's session has no bob messages
alice_msgs = [m['content'] for m in alice_session.get_messages()]
assert not any("Bob" in m for m in alice_msgs), "Session isolation broken!"
print("\nSession isolation: PASS")

## Part 5: Inspect in RedisInsight

Open http://localhost:8001 and:
1. Go to **Browser** tab
2. Filter by key pattern `chat:*` to see all session keys
3. Click on a List key to see the serialized messages
4. Click on a Hash key to see session metadata
5. Note the TTL countdown on each key

In [ ]:
# List all chat keys currently in Redis
print("All chat keys in Redis:")
for key in sorted(r.scan_iter("chat:*")):
    ttl = r.ttl(key)
    key_type = r.type(key)
    count = r.llen(key) if key_type == "list" else r.hlen(key) if key_type == "hash" else "N/A"
    print(f"  {key:<55} type={key_type:<8} items={count:<5} TTL={ttl}s")

## Cleanup

In [ ]:
# Clean up this notebook's test keys
patterns = [f"chat:{session_id}*", f"chat:{lc_session_id}*", "chat:demo-session-001*",
            "chat:alice-multi-test*", "chat:bob-multi-test*"]
for pat in patterns:
    keys = list(r.scan_iter(pat))
    if keys:
        r.delete(*keys)
        print(f"Deleted {len(keys)} key(s) matching {pat}")

## Summary

| Approach | Storage | Best For |
|----------|---------|----------|
| Raw redis-py (List + Hash) | List of JSON strings | Full control, custom schemas |
| LangChain RedisChatMessageHistory | JSON docs via RedisVL | LangChain chains, production |

Both support:
- Sliding TTL (session expiry on inactivity)
- Message window (last N turns for LLM context)
- Multi-session isolation (each session has its own key)

Next: **02_vector_store/02_hands_on.ipynb** — Add TechBot's knowledge base (RAG retrieval).